# 🌊 Flash Flood Prediction in Hilly Regions (Himachal Pradesh)
### Smart India Hackathon — End-to-End Machine Learning & Hydrological Analysis

This notebook demonstrates the complete end-to-end data pipeline, hydrological feature engineering, ground-truth disaster labeling, model benchmarking on a strict chronological split (2015-2022 Train vs 2023-2025 Test), and inference explainability.

In [ ]:
import os
import sys
from pathlib import Path
import pandas as pd
import numpy as np
import joblib
import matplotlib.pyplot as plt

# Set project root
PROJECT_ROOT = Path(os.getcwd()).resolve().parent
sys.path.append(str(PROJECT_ROOT))

from src.utils.config import LABELED_DATA_PATH, MODEL_PATH, ML_FEATURE_COLS, TARGET_COL
print("Project Root:", PROJECT_ROOT)
print("Model Path:", MODEL_PATH)

## 1. Data Ingestion & Ground-Truth Inspection
The dataset integrates 11 years (2015–2025) of NASA POWER MERRA-2 meteorological telemetry with confirmed historical Himachal Pradesh disaster chronicles and IMD Flash Flood Guidance thresholds.

In [ ]:
df = pd.read_csv(LABELED_DATA_PATH)
print("Dataset Shape:", df.shape)
print("Class Distribution:")
print(df[TARGET_COL].value_counts(normalize=True).rename({0: "Normal Days", 1: "Flood Event Days"}))
df.head()

## 2. Antecedent Soil Moisture & Extreme Precipitation Patterns
Examine the correlation between daily precipitation (`PRECTOTCORR`) and 3-day antecedent rainfall (`RAIN_3DAY`).

In [ ]:
plt.figure(figsize=(10, 6))
plt.scatter(df[df[TARGET_COL] == 0]['PRECTOTCORR'], df[df[TARGET_COL] == 0]['RAIN_3DAY'], color='gray', alpha=0.3, label='Normal Days')
plt.scatter(df[df[TARGET_COL] == 1]['PRECTOTCORR'], df[df[TARGET_COL] == 1]['RAIN_3DAY'], color='red', alpha=0.8, s=60, label='Flood Event Days')
plt.axvline(64.5, color='blue', linestyle='--', label='IMD Heavy Rain Threshold (64.5 mm)')
plt.xlabel('Daily Precipitation (mm)')
plt.ylabel('3-Day Cumulative Rainfall (mm)')
plt.title('Flash Flood Triggers: Precipitation vs Antecedent Saturation')
plt.legend()
plt.grid(True, alpha=0.3)
plt.show()

## 3. Load Trained Model & Inspect Performance Benchmark
Load the serialized model artifact trained on the strict chronological split (Train: 2015-2022, Test: 2023-2025).

In [ ]:
artifact = joblib.load(MODEL_PATH)
print('Model Architecture:', artifact['model_name'])
print('\nTest Set Metrics (2023-2025 Period):')
for metric, val in artifact['metrics'].items():
    print(f'  {metric:<20}: {val:.4f}')

print('\nConfusion Matrix:', artifact['confusion_matrix'])

## 4. Feature Importance & Risk Attribution
Identify which hydrometeorological features drive the model decisions.

In [ ]:
feat_imp = pd.Series(artifact['feature_importances']).sort_values(ascending=True)
plt.figure(figsize=(10, 6))
feat_imp.plot(kind='barh', color='steelblue')
plt.title('Permutation Feature Importances for Flash Flood Prediction')
plt.xlabel('Relative Importance Weight')
plt.grid(True, alpha=0.3)
plt.show()

## 5. Live Inference Verification
Test the production `FloodPredictor` inference engine on an extreme storm scenario vs benign dry weather.

In [ ]:
from src.ml.predict import FloodPredictor
predictor = FloodPredictor()

test_storm = {
    'PRECTOTCORR': 126.85, # July 10, 2023 disaster reading
    'T2M': 21.0,
    'RH2M': 92.5,
    'WS2M': 3.5,
    'RAIN_3DAY': 302.0,
    'RAIN_7DAY': 355.0,
}

result = predictor.predict_single(test_storm)
print('\n--- Extreme Storm Prediction ---')
print('Predicted Risk Level :', result['risk_level'])
print('Flood Probability    :', f"{result['probability_pct']}%")
print('Top Driving Factors  :')
for f in result['top_contributing_factors']:
    print(f"  - {f['feature']}: {f['value']} (Weight: {f['importance']})")